# 12 — Embedding migration (multilingual)

Re-embeds the catalog with a **multilingual** sentence model so non-English books stop forming a
degenerate same-language cluster, then rebuilds the artifacts that consume the vectors. The
English-only `bge-small-en` cannot read other scripts (e.g. Cyrillic), so every foreign book
collapses to ~0.95+ cosine with the other foreign books — which lets a single off-distribution
seed hijack the feed. A multilingual encoder embeds by *meaning* across languages.

**What this notebook regenerates** (writes to `artifacts/`, downloadable from `/kaggle/working`):
- `embeddings.npy` — the catalog re-encoded with the new model (row order == `catalog.parquet`).
- `faiss.idx` — cosine index rebuilt over the new vectors (LLM retrieval).
- `models.joblib` — the full model zoo rebuilt (same as `07_models`) so the embedding-based
  recommenders (`content_emb`, `similar`, `maxsim`, the `content` half of the hybrid) match the
  new vectors.

**What does NOT change** (these never saw the text embeddings, so they need no re-train): the
collaborative models (`popularity`, `svd`, Mult-VAE), the sequential models, and the TF-IDF / BoW
content baselines. They are re-fit here only to keep one consistent `models.joblib`; their results
are unchanged. The Mult-VAE checkpoint (`multvae_last.pt`) is independent and is not touched.


## Kaggle bootstrap
Pulls `book_recsys` (from **master**) and links the dataset inputs into `artifacts/` (writable,
under `/kaggle/working`). We link only `sample.parquet` + `catalog.parquet` — **not** the old
`embeddings.npy`, since this notebook regenerates it. Skipped locally.

In [ ]:
import os
if os.path.exists('/kaggle'):
    import sys, glob
    REPO = '/kaggle/working/book_recsys'
    if not os.path.exists(REPO):
        !git clone -b master https://github.com/MayaDeneva/book_recsys.git {REPO}
    else:
        !cd {REPO} && git pull origin master
    !pip install -e {REPO} -q
    sys.path.insert(0, REPO)
    os.chdir(REPO)
    os.makedirs('artifacts', exist_ok=True)
    # NB: deliberately NOT linking embeddings.npy — we re-encode it below.
    for _f in ['sample.parquet', 'catalog.parquet']:
        _src = glob.glob(f'/kaggle/input/**/{_f}', recursive=True)
        if _src and not os.path.exists(f'artifacts/{_f}'):
            os.symlink(_src[0], f'artifacts/{_f}')
    print('Kaggle: repo at', REPO, '| artifacts:', sorted(os.listdir('artifacts')))
else:
    print('not on Kaggle — running locally with artifacts/ in place')

In [ ]:
!pip install -q sentence-transformers faiss-cpu

## Config — pick the multilingual model

`paraphrase-multilingual-MiniLM-L12-v2` is **384-d** — a drop-in for the current `bge-small-en`
pipeline (no dimension plumbing) and it covers 50+ languages incl. Bulgarian. Swap in a stronger
model if you want; mind the dimension and the e5 prefix note below.

| model | dim | notes |
|---|---|---|
| `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` | 384 | drop-in, no prefixes |
| `sentence-transformers/LaBSE` | 768 | translation-pair optimized (most language-agnostic) |
| `BAAI/bge-m3` | 1024 | strong multilingual, no prefixes |
| `intfloat/multilingual-e5-large` | 1024 | strong, **needs** `passage:` / `query:` prefixes |

The query encoder at serving time is `config.EMBED_MODEL` (env `BOOK_RECSYS_EMBED_MODEL`) and
**must equal `EMBED_MODEL` below**. If you pick an e5 model, the live retriever must also prefix
queries with `query: ` (a code change in `llm/retrieve.py`); the non-e5 models above need none.

In [ ]:
EMBED_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
BATCH = 256                       # raise on a Kaggle GPU; lower if you hit OOM
PASSAGE_PREFIX = 'passage: ' if 'e5' in EMBED_MODEL.lower() else ''   # e5 needs it; others don't
print('encoder:', EMBED_MODEL, '| passage prefix:', repr(PASSAGE_PREFIX))

In [ ]:
import glob, os, joblib
import numpy as np, pandas as pd, torch
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer
import faiss

from book_recsys.data.splits import leave_last_n_out
from book_recsys.data.filters import filter_min_rating
from book_recsys.features.document import build_documents
from book_recsys.features.vectorize import tfidf_matrix
from book_recsys.features.index import build_index
from book_recsys.models.classical.popularity import PopularityRecommender
from book_recsys.models.classical.svd import SvdRecommender
from book_recsys.models.content.content import ContentRecommender
from book_recsys.models.content.similar import SimilarItemsRecommender
from book_recsys.models.content.maxsim import MaxSimRecommender
from book_recsys.models.hybrid.learned import LearnedHybridRecommender

def find(name):
    for p in (f'artifacts/{name}', f'../artifacts/{name}'):
        if glob.glob(p): return glob.glob(p)[0]
    raise FileNotFoundError(name)

catalog = pd.read_parquet(find('catalog.parquet'))
sample = pd.read_parquet(find('sample.parquet'))
book_ids = catalog['book_id'].tolist()
ART = os.path.dirname(find('catalog.parquet'))   # write everything beside the inputs
print(len(book_ids), 'books ·', len(sample), 'interactions · artifacts ->', os.path.abspath(ART))

### Step 1 — re-encode the catalog

`catalog.parquet` row order is preserved, so `embeddings.npy` row *i* stays book `book_ids[i]` —
the recommenders rely on this. We encode directly (not via the cached `embed_documents`) so we can
set the batch size and **force a fresh** computation, overwriting the old `bge-small-en` vectors.

In [ ]:
docs = build_documents(catalog)
if PASSAGE_PREFIX:
    docs = [PASSAGE_PREFIX + d for d in docs]
print('sample document:\n', docs[0][:200], '\n')

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('encoding', len(docs), 'documents on', device)
model = SentenceTransformer(EMBED_MODEL, device=device)
emb = model.encode(docs, batch_size=BATCH, show_progress_bar=True,
                   convert_to_numpy=True, normalize_embeddings=False)
emb = np.ascontiguousarray(emb, dtype='float32')
np.save(f'{ART}/embeddings.npy', emb)
assert emb.shape[0] == len(book_ids), 'row count must match the catalog'
print('saved embeddings.npy', emb.shape, '(was 384-d bge-small-en)')

### Step 2 — rebuild the FAISS index (LLM retrieval)

In [ ]:
index = build_index(emb)                 # inner-product over L2-normalized rows = cosine
faiss.write_index(index, f'{ART}/faiss.idx')
print('faiss.idx rebuilt · ntotal =', index.ntotal, '· dim =', index.d)

### Step 3 — rebuild `models.joblib`

Identical to `07_models`, but over the new vectors. The embedding-based models (`content_emb`,
`similar`, `maxsim`, and the `content` scorer inside the hybrid) now reflect the multilingual
space; the CF / TF-IDF models are re-fit unchanged so the saved zoo stays self-consistent.

In [ ]:
train, _ = leave_last_n_out(sample, n=1)   # same split eval holds out from

# classical: popularity + collaborative filtering (no text -> identical to before)
popularity = PopularityRecommender().fit(train)
svd = SvdRecommender(n_factors=64).fit(train)
svd_pos = SvdRecommender(n_factors=64).fit(filter_min_rating(train, 4))

# content: TF-IDF (own representation, unaffected) + the NEW multilingual embeddings
tfidf, _ = tfidf_matrix(build_documents(catalog), max_features=3000)
content_tfidf = ContentRecommender(book_ids, tfidf).fit()
content_emb = ContentRecommender(book_ids, emb).fit()
similar = SimilarItemsRecommender(book_ids, emb).fit()   # UC4 item-item
maxsim = MaxSimRecommender(book_ids, emb).fit()          # UC1 max-sim to any liked book

# hybrid: learned reranker over CF + content (re-fit so its content feature matches the new emb)
users = train['user_id'].unique()
fit_users = np.random.default_rng(0).choice(users, size=min(5000, len(users)), replace=False)
hybrid_train = train[train['user_id'].isin(fit_users)]
hybrid = LearnedHybridRecommender({'cf': svd, 'content': content_emb}, seed=0).fit(hybrid_train)
hybrid_pop = LearnedHybridRecommender(
    {'cf': svd, 'content': content_emb, 'pop': popularity}, seed=0).fit(hybrid_train)
hybrid_popneg = LearnedHybridRecommender(
    {'cf': svd, 'content': content_emb}, seed=0, neg_sampling='popularity').fit(hybrid_train)

models = {
    'popularity': popularity, 'svd': svd, 'svd_rating>=4': svd_pos,
    'content_tfidf_full': content_tfidf, 'content_emb': content_emb,
    'hybrid_cf_content': hybrid, 'hybrid_cf_content_pop': hybrid_pop,
    'hybrid_cf_content_popneg': hybrid_popneg,
    'maxsim': maxsim, 'similar': similar,
}
joblib.dump(models, f'{ART}/models.joblib')
print('saved', list(models), '->', os.path.abspath(f'{ART}/models.joblib'))

### Step 4 — sanity: did the degenerate cluster break up?

Two checks tied to the root cause: (1) the most common non-English language should no longer be a
near-1.0 cosine blob, and (2) a seed's nearest neighbours should look topical, with sane language
tags rather than a wall of one foreign language.

In [ ]:
pos = {b: i for i, b in enumerate(book_ids)}
V = normalize(emb)

def avg_off_diag_cosine(ids, n=300):
    rows = [pos[b] for b in ids[:n] if b in pos]
    S = V[rows] @ V[rows].T
    m = len(rows)
    return float((S.sum() - m) / (m * m - m))

norm_lang = catalog['language_code'].fillna('').astype(str).str.lower().str.split('-').str[0].str[:3]
foreign = norm_lang[~norm_lang.isin(['eng', 'en', ''])].value_counts()
print('top non-English languages:', dict(foreign.head(5)), '\n')
pick = foreign.index[0]
ids = catalog.loc[norm_lang == pick, 'book_id'].tolist()
print(f"avg pairwise cosine among {min(len(ids),300)} '{pick}' books: "
      f"{avg_off_diag_cosine(ids):.3f}  (near 1.0 = degenerate; lower = healthy)\n")

id2title = dict(zip(catalog['book_id'], catalog['title']))
seed_book = train['book_id'].value_counts().index[0]
print('anchor:', id2title.get(seed_book), '|', norm_lang[pos[seed_book]])
for b in similar.recommend(seed_book, 8):
    print('   ', norm_lang[pos[b]], '·', id2title.get(b, b))

### Step 5 — use the new artifacts

1. **Download** `embeddings.npy`, `faiss.idx`, `models.joblib` from `/kaggle/working` (or *Save
   Version* and add this kernel's output as a dataset).
2. Point the app/eval at them (replace the files in `artifacts/`).
3. **Set the serving encoder to match:** `export BOOK_RECSYS_EMBED_MODEL='<EMBED_MODEL above>'`
   before launching `uvicorn book_recsys.api.app:get_app --factory`. (For an e5 model, also add the
   `query: ` prefix in `llm/retrieve.py`.)
4. **Re-run `08_evaluation`** to refresh only the embedding-based rows (`content_emb`, `maxsim`,
   `similar`, hybrid). The `popularity` / `svd` / Mult-VAE / TF-IDF rows are unchanged.

The collaborative and TF-IDF baselines did not need re-training; only the text vectors and the
non-parametric indexes built on them were regenerated.